# Walkthrough: Introduction to NLP & Text Analysis

In this walkthrough, we'll explore Natural Language Processing hands-on. We'll see:

1. **Why NLP is hard** — ambiguity, sarcasm, and slang that break computers
2. **The NLP pipeline** — cleaning, tokenizing, and preparing text for analysis
3. **Feature extraction** — converting words into numbers a model can understand
4. **Sentiment analysis** — building a working classifier and testing its limits

Everything we learn today is building toward **Transformer architecture** — the engine behind ChatGPT, BERT, and modern AI.

---

## Setup

Let's install and import everything we'll need.

In [1]:
# Install required packages (run once)
# !pip install nltk scikit-learn -q

In [2]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import pandas as pd

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print("All imports and downloads complete!")

All imports and downloads complete!


---

# Part 1: Why NLP is Hard

Before we build anything, let's see why this problem is so fascinating — and so difficult.

## 1.1 The Stress Test

In English, stressing different words in a sentence changes its meaning entirely. But to a computer, the text is **identical** every time.

In [3]:
# These all mean VERY different things when spoken aloud:
meanings = {
    "*I* never said she stole my money":     "Someone else said it, not me.",
    "I *never* said she stole my money":      "I absolutely did not say it.",
    "I never *said* she stole my money":      "I didn't SAY it — maybe I implied it.",
    "I never said *she* stole my money":      "I said someone stole it, just not her.",
    "I never said she *stole* my money":      "She has my money, but maybe she borrowed it.",
    "I never said she stole *my* money":      "She stole money, just not mine.",
    "I never said she stole my *money*":      "She stole something of mine, but not money.",
}

print("7 different meanings from the SAME 7 words:\n")
for emphasis, meaning in meanings.items():
    print(f"  {emphasis}")
    print(f"    → {meaning}\n")

# But to Python, they're all the same:
base = "I never said she stole my money"
print(f"\nTo a computer, the raw text is always: '{base}'")
print(f"No emphasis, no tone, no context. Just characters.")

7 different meanings from the SAME 7 words:

  *I* never said she stole my money
    → Someone else said it, not me.

  I *never* said she stole my money
    → I absolutely did not say it.

  I never *said* she stole my money
    → I didn't SAY it — maybe I implied it.

  I never said *she* stole my money
    → I said someone stole it, just not her.

  I never said she *stole* my money
    → She has my money, but maybe she borrowed it.

  I never said she stole *my* money
    → She stole money, just not mine.

  I never said she stole my *money*
    → She stole something of mine, but not money.


To a computer, the raw text is always: 'I never said she stole my money'
No emphasis, no tone, no context. Just characters.


## 1.2 One Word, Many Meanings

The word "bank" appears in both of these sentences. A simple search finds it in both — but the meanings are completely different.

In [4]:
sentences = [
    "I deposited my paycheck at the bank this morning.",
    "We sat on the bank of the river and watched the sunset.",
    "You can bank on me to get the job done.",
    "The airplane made a sharp bank to the left.",
]

print("Searching for the word 'bank' in each sentence:\n")
for sentence in sentences:
    found = "bank" in sentence.lower()
    print(f"  ✓ Found 'bank' in: \"{sentence}\"")

print(f"\nPython found 'bank' in ALL {len(sentences)} sentences.")
print("But each one uses 'bank' with a completely different meaning!")
print("\nA computer matches the characters 'b-a-n-k' — it has no idea about meaning.")

Searching for the word 'bank' in each sentence:

  ✓ Found 'bank' in: "I deposited my paycheck at the bank this morning."
  ✓ Found 'bank' in: "We sat on the bank of the river and watched the sunset."
  ✓ Found 'bank' in: "You can bank on me to get the job done."
  ✓ Found 'bank' in: "The airplane made a sharp bank to the left."

Python found 'bank' in ALL 4 sentences.
But each one uses 'bank' with a completely different meaning!

A computer matches the characters 'b-a-n-k' — it has no idea about meaning.


## 1.3 Sarcasm Breaks Everything

Let's build a super simple sentiment detector that counts positive and negative words. Then let's watch it fail spectacularly on sarcasm.

In [5]:
# A naive keyword-based sentiment detector
positive_words = {"wonderful", "great", "love", "amazing", "beautiful", "fantastic", "excellent", "good", "best", "happy"}
negative_words = {"terrible", "awful", "hate", "worst", "bad", "horrible", "boring", "waste", "disappointing", "ugly"}

def naive_sentiment(text):
    words = text.lower().split()
    pos_count = sum(1 for w in words if w in positive_words)
    neg_count = sum(1 for w in words if w in negative_words)
    
    if pos_count > neg_count:
        return "POSITIVE", pos_count, neg_count
    elif neg_count > pos_count:
        return "NEGATIVE", pos_count, neg_count
    else:
        return "NEUTRAL", pos_count, neg_count

# Test on straightforward cases first
easy_cases = [
    "This movie was absolutely fantastic and amazing!",
    "Terrible film, worst I've ever seen, truly awful.",
]

print("=== Easy Cases (these work fine) ===\n")
for text in easy_cases:
    sentiment, pos, neg = naive_sentiment(text)
    print(f"  \"{text}\"")
    print(f"  → {sentiment} (positive words: {pos}, negative words: {neg})\n")

# Now the sarcasm...
sarcastic_cases = [
    "Oh wonderful, my flight is delayed again.",
    "Sure, I'd love to work this weekend. Great.",
    "What a beautiful day to be stuck in traffic.",
    "That was the best worst movie I've ever seen.",
]

print("\n=== Sarcastic Cases (watch it fail) ===\n")
for text in sarcastic_cases:
    sentiment, pos, neg = naive_sentiment(text)
    print(f"  \"{text}\"")
    print(f"  → {sentiment} (positive words: {pos}, negative words: {neg})")
    print(f"  ❌ This is actually NEGATIVE — the model got fooled by positive-sounding words!\n")

=== Easy Cases (these work fine) ===

  "This movie was absolutely fantastic and amazing!"
  → POSITIVE (positive words: 1, negative words: 0)

  "Terrible film, worst I've ever seen, truly awful."
  → NEGATIVE (positive words: 0, negative words: 2)


=== Sarcastic Cases (watch it fail) ===

  "Oh wonderful, my flight is delayed again."
  → NEUTRAL (positive words: 0, negative words: 0)
  ❌ This is actually NEGATIVE — the model got fooled by positive-sounding words!

  "Sure, I'd love to work this weekend. Great."
  → POSITIVE (positive words: 1, negative words: 0)
  ❌ This is actually NEGATIVE — the model got fooled by positive-sounding words!

  "What a beautiful day to be stuck in traffic."
  → POSITIVE (positive words: 1, negative words: 0)
  ❌ This is actually NEGATIVE — the model got fooled by positive-sounding words!

  "That was the best worst movie I've ever seen."
  → NEUTRAL (positive words: 1, negative words: 1)
  ❌ This is actually NEGATIVE — the model got fooled by positi

## 1.4 Slang Evolution

Language evolves constantly. A model trained on data from a few years ago wouldn't understand today's slang at all.

In [6]:
# A dictionary a model might have learned from 2020 training data
known_vocabulary = {
    "good": "positive adjective",
    "bad": "negative adjective",
    "fire": "combustion, flames, dangerous",
    "sick": "ill, unwell, needs medical attention",
    "cap": "a hat, a lid, a cover",
    "mid": "middle, prefix meaning halfway",
    "slay": "to kill violently",
    "bet": "a wager, gambling",
}

# What people actually mean in 2026
modern_slang = [
    ("That outfit is fire 🔥",           "fire",          "Amazing/excellent"),
    ("This movie is so mid",              "mid",           "Mediocre/average"),
    ("No cap, that was the best meal",    "cap",           "No lie / I'm serious"),
    ("She absolutely slayed that exam",   "slay",          "Dominated / did amazing"),
    ("He's mogging everyone at the gym",  "mogging",       "Outshining everyone in appearance"),
    ("She's been looksmaxxing all year",  "looksmaxxing",  "Maximizing physical appearance"),
    ("That fit is a 6-7 at best",         "6-7",           "Rating on an attractiveness scale"),
]

print("How a 2020-trained model would interpret modern slang:\n")
for sentence, word, actual_meaning in modern_slang:
    old_meaning = known_vocabulary.get(word, "❓ UNKNOWN — not in vocabulary")
    print(f"  Sentence: \"{sentence}\"")
    print(f"  Model thinks '{word}' means: {old_meaning}")
    print(f"  Actually means: {actual_meaning}")
    print()

How a 2020-trained model would interpret modern slang:

  Sentence: "That outfit is fire 🔥"
  Model thinks 'fire' means: combustion, flames, dangerous
  Actually means: Amazing/excellent

  Sentence: "This movie is so mid"
  Model thinks 'mid' means: middle, prefix meaning halfway
  Actually means: Mediocre/average

  Sentence: "No cap, that was the best meal"
  Model thinks 'cap' means: a hat, a lid, a cover
  Actually means: No lie / I'm serious

  Sentence: "She absolutely slayed that exam"
  Model thinks 'slay' means: to kill violently
  Actually means: Dominated / did amazing

  Sentence: "He's mogging everyone at the gym"
  Model thinks 'mogging' means: ❓ UNKNOWN — not in vocabulary
  Actually means: Outshining everyone in appearance

  Sentence: "She's been looksmaxxing all year"
  Model thinks 'looksmaxxing' means: ❓ UNKNOWN — not in vocabulary
  Actually means: Maximizing physical appearance

  Sentence: "That fit is a 6-7 at best"
  Model thinks '6-7' means: ❓ UNKNOWN — not i

---

# Part 2: The NLP Pipeline

**Scenario:** You're a data scientist at a company that wants to analyze customer product reviews. Raw reviews are messy — full of typos, emojis, random capitalization, and noise. We need to clean them up before any model can learn from them.

Let's build a complete text processing pipeline, step by step.

## 2.1 Raw Text is Messy

Here's what real customer reviews look like. This is what a computer has to work with.

In [7]:
raw_reviews = [
    "OMG this phone is soooo good!!! 📱📱📱 battery lasts 4EVER lol 10/10 would recomend!!!",
    "Worst. Purchase. EVER!!! 😤😤 the screen craked after 2 days & customer service was USELESS @company #neveragain",
    "its ok i guess... nothing special tbh. camra is decent but not $800 decent lmaooo",
    "LOVE LOVE LOVE this product!!! My skin has never looked better 💕✨ #skincare #blessed",
    "DO NOT BUY!! broke after 1 week. total waste of $$$$. returnd it ASAP smh 👎👎👎",
]

print("Raw customer reviews — this is what we're working with:\n")
for i, review in enumerate(raw_reviews, 1):
    print(f"  Review {i}: {review}\n")

print("\nNotice: emojis, ALL CAPS, typos, abbreviations, hashtags, @mentions, repeated punctuation...")
print("A model can't learn from this mess. We need to clean it up.")

Raw customer reviews — this is what we're working with:

  Review 1: OMG this phone is soooo good!!! 📱📱📱 battery lasts 4EVER lol 10/10 would recomend!!!

  Review 2: Worst. Purchase. EVER!!! 😤😤 the screen craked after 2 days & customer service was USELESS @company #neveragain

  Review 3: its ok i guess... nothing special tbh. camra is decent but not $800 decent lmaooo

  Review 4: LOVE LOVE LOVE this product!!! My skin has never looked better 💕✨ #skincare #blessed

  Review 5: DO NOT BUY!! broke after 1 week. total waste of $$$$. returnd it ASAP smh 👎👎👎


Notice: emojis, ALL CAPS, typos, abbreviations, hashtags, @mentions, repeated punctuation...
A model can't learn from this mess. We need to clean it up.


## 2.2 Step 1: Lowercasing

To a computer, "AMAZING", "Amazing", and "amazing" are three completely different strings. Let's fix that.

In [8]:
# To Python, these are NOT the same:
print("Are these the same word?")
print(f"  'AMAZING' == 'amazing' → {('AMAZING' == 'amazing')}")
print(f"  'Amazing' == 'amazing' → {('Amazing' == 'amazing')}")
print(f"  'AMAZING' == 'Amazing' → {('AMAZING' == 'Amazing')}")

# Lowercasing unifies them
print("\nAfter .lower():")
print(f"  'AMAZING'.lower() == 'amazing'.lower() → {'AMAZING'.lower() == 'amazing'.lower()}")

# Apply to a review
sample = raw_reviews[0]
print(f"\nBefore: {sample}")
print(f"After:  {sample.lower()}")

Are these the same word?
  'AMAZING' == 'amazing' → False
  'Amazing' == 'amazing' → False
  'AMAZING' == 'Amazing' → False

After .lower():
  'AMAZING'.lower() == 'amazing'.lower() → True

Before: OMG this phone is soooo good!!! 📱📱📱 battery lasts 4EVER lol 10/10 would recomend!!!
After:  omg this phone is soooo good!!! 📱📱📱 battery lasts 4ever lol 10/10 would recomend!!!


## 2.3 Step 2: Removing Punctuation & Special Characters

Emojis, hashtags, @mentions, and excessive punctuation are noise. Let's strip them out using regular expressions.

In [9]:
def remove_noise(text):
    """Remove everything that isn't a letter or space."""
    # Keep only letters and spaces
    cleaned = re.sub(r'[^a-zA-Z\s]', '', text)
    # Remove extra whitespace
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    return cleaned

print("Removing punctuation, emojis, numbers, and special characters:\n")
for i, review in enumerate(raw_reviews, 1):
    cleaned = remove_noise(review.lower())
    print(f"  Review {i}:")
    print(f"    Before: {review}")
    print(f"    After:  {cleaned}\n")

Removing punctuation, emojis, numbers, and special characters:

  Review 1:
    Before: OMG this phone is soooo good!!! 📱📱📱 battery lasts 4EVER lol 10/10 would recomend!!!
    After:  omg this phone is soooo good battery lasts ever lol would recomend

  Review 2:
    Before: Worst. Purchase. EVER!!! 😤😤 the screen craked after 2 days & customer service was USELESS @company #neveragain
    After:  worst purchase ever the screen craked after days customer service was useless company neveragain

  Review 3:
    Before: its ok i guess... nothing special tbh. camra is decent but not $800 decent lmaooo
    After:  its ok i guess nothing special tbh camra is decent but not decent lmaooo

  Review 4:
    Before: LOVE LOVE LOVE this product!!! My skin has never looked better 💕✨ #skincare #blessed
    After:  love love love this product my skin has never looked better skincare blessed

  Review 5:
    Before: DO NOT BUY!! broke after 1 week. total waste of $$$$. returnd it ASAP smh 👎👎👎
    After:

## 2.4 Tokenization

Tokenization splits text into individual words ("tokens"). We can use simple `.split()` or NLTK's smarter tokenizer.

In [10]:
# Simple split vs NLTK tokenizer
sample_text = "I can't believe they wouldn't let us in. It's ridiculous!"

# Method 1: Simple split (naive)
simple_tokens = sample_text.split()
print("Method 1 — str.split():")
print(f"  {simple_tokens}")
print(f"  Notice: 'can't' stays as one token, 'in.' includes the period\n")

# Method 2: NLTK word_tokenize (smarter)
nltk_tokens = word_tokenize(sample_text)
print("Method 2 — NLTK word_tokenize():")
print(f"  {nltk_tokens}")
print(f"  Notice: 'can't' → 'ca' + 'n't', punctuation separated\n")

# Apply to our cleaned review
cleaned_review = remove_noise(raw_reviews[0].lower())
tokens = cleaned_review.split()
print(f"Our cleaned review tokenized:")
print(f"  Text: \"{cleaned_review}\"")
print(f"  Tokens: {tokens}")
print(f"  Count: {len(tokens)} words")

Method 1 — str.split():
  ['I', "can't", 'believe', 'they', "wouldn't", 'let', 'us', 'in.', "It's", 'ridiculous!']
  Notice: 'can't' stays as one token, 'in.' includes the period

Method 2 — NLTK word_tokenize():
  ['I', 'ca', "n't", 'believe', 'they', 'would', "n't", 'let', 'us', 'in', '.', 'It', "'s", 'ridiculous', '!']
  Notice: 'can't' → 'ca' + 'n't', punctuation separated

Our cleaned review tokenized:
  Text: "omg this phone is soooo good battery lasts ever lol would recomend"
  Tokens: ['omg', 'this', 'phone', 'is', 'soooo', 'good', 'battery', 'lasts', 'ever', 'lol', 'would', 'recomend']
  Count: 12 words


## 2.5 Stopword Removal

Stopwords are common words like "the", "is", "at" that appear everywhere but carry little meaning. Removing them focuses on the important words.

**But be careful** — sometimes removing stopwords changes the meaning entirely!

In [11]:
stop_words = set(stopwords.words('english'))

# Let's see what stopwords look like
print(f"NLTK has {len(stop_words)} English stopwords. Here are some:")
print(f"  {sorted(list(stop_words))[:20]}...\n")

# Apply stopword removal
sample = "this phone is so good the battery lasts forever and i would recommend it"
tokens = sample.split()
filtered = [word for word in tokens if word not in stop_words]

print(f"Before: {sample}")
print(f"After:  {' '.join(filtered)}")
print(f"Removed: {[w for w in tokens if w in stop_words]}\n")

# THE DANGER: negation
print("=" * 60)
print("⚠️  THE DANGER OF STOPWORD REMOVAL:\n")

dangerous_examples = [
    "The movie was not good at all",
    "I do not recommend this product",
    "This is not what I expected",
]

for text in dangerous_examples:
    tokens = text.lower().split()
    filtered = [w for w in tokens if w not in stop_words]
    print(f"  Before: \"{text}\"")
    print(f"  After:  \"{' '.join(filtered)}\"")
    print(f"  😬 We lost 'not' — the meaning completely flipped!\n")

NLTK has 198 English stopwords. Here are some:
  ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']...

Before: this phone is so good the battery lasts forever and i would recommend it
After:  phone good battery lasts forever would recommend
Removed: ['this', 'is', 'so', 'the', 'and', 'i', 'it']

⚠️  THE DANGER OF STOPWORD REMOVAL:

  Before: "The movie was not good at all"
  After:  "movie good"
  😬 We lost 'not' — the meaning completely flipped!

  Before: "I do not recommend this product"
  After:  "recommend product"
  😬 We lost 'not' — the meaning completely flipped!

  Before: "This is not what I expected"
  After:  "expected"
  😬 We lost 'not' — the meaning completely flipped!



## 2.6 Stemming vs Lemmatization

Both reduce words to a root form, but they work very differently.

- **Stemming** = crude chopping (fast but sometimes produces nonsense)
- **Lemmatization** = smart dictionary lookup (slower but produces real words)

In [12]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

test_words = ["running", "better", "studies", "caring", "flies", "happily", "organization", "fairly"]

print(f"{'Original':<15} {'Stemmed':<15} {'Lemmatized':<15}")
print("-" * 45)

for word in test_words:
    stemmed = stemmer.stem(word)
    lemmatized = lemmatizer.lemmatize(word)
    
    # Flag bad stems
    flag = " ← yikes!" if stemmed != lemmatized and len(stemmed) < len(word) - 3 else ""
    print(f"{word:<15} {stemmed:<15} {lemmatized:<15}{flag}")

print("\nNotice how stemming turns 'caring' → 'car' and 'organization' → 'organ'!")
print("Lemmatization is smarter — it uses a dictionary to find the real root word.")

Original        Stemmed         Lemmatized     
---------------------------------------------
running         run             running         ← yikes!
better          better          better         
studies         studi           study          
caring          care            caring         
flies           fli             fly            
happily         happili         happily        
organization    organ           organization    ← yikes!
fairly          fairli          fairly         

Notice how stemming turns 'caring' → 'car' and 'organization' → 'organ'!
Lemmatization is smarter — it uses a dictionary to find the real root word.


## 2.7 The Complete Pipeline

Let's combine all the steps into one reusable function and process our reviews.

In [13]:
def preprocess_text(text):
    """Complete NLP preprocessing pipeline."""
    # Step 1: Lowercase
    text = text.lower()
    
    # Step 2: Remove special characters (keep only letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Step 3: Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Step 4: Tokenize
    tokens = text.split()
    
    # Step 5: Remove stopwords (keeping 'not' for sentiment!)
    stop_words = set(stopwords.words('english')) - {'not', 'no', 'nor', 'neither', 'never'}
    tokens = [word for word in tokens if word not in stop_words]
    
    # Step 6: Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return ' '.join(tokens)


print("Complete pipeline applied to our raw reviews:\n")
for i, review in enumerate(raw_reviews, 1):
    processed = preprocess_text(review)
    print(f"Review {i}:")
    print(f"  Raw:       {review}")
    print(f"  Processed: {processed}\n")

print("\nFrom messy, noisy text → clean, focused words ready for analysis!")

Complete pipeline applied to our raw reviews:

Review 1:
  Raw:       OMG this phone is soooo good!!! 📱📱📱 battery lasts 4EVER lol 10/10 would recomend!!!
  Processed: omg phone soooo good battery last ever lol would recomend

Review 2:
  Raw:       Worst. Purchase. EVER!!! 😤😤 the screen craked after 2 days & customer service was USELESS @company #neveragain
  Processed: worst purchase ever screen craked day customer service useless company neveragain

Review 3:
  Raw:       its ok i guess... nothing special tbh. camra is decent but not $800 decent lmaooo
  Processed: ok guess nothing special tbh camra decent not decent lmaooo

Review 4:
  Raw:       LOVE LOVE LOVE this product!!! My skin has never looked better 💕✨ #skincare #blessed
  Processed: love love love product skin never looked better skincare blessed

Review 5:
  Raw:       DO NOT BUY!! broke after 1 week. total waste of $$$$. returnd it ASAP smh 👎👎👎
  Processed: not buy broke week total waste returnd asap smh


From messy, no

---

# Part 3: Feature Extraction

Machine learning models need **numbers**, not words. How do we convert text into a format a model can learn from?

**Scenario:** We have cleaned reviews. Now we need to turn them into numerical features.

## 3.1 Bag of Words

The simplest approach: count how many times each word appears in each document.

In [14]:
# Simple example documents
documents = [
    "I love this movie it is great",
    "This movie is terrible I hate it",
    "Great movie with great acting",
]

# Create Bag of Words
vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(documents)

# Display as a readable table
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(documents))]
)

print("Documents:")
for i, doc in enumerate(documents, 1):
    print(f"  Doc {i}: \"{doc}\"")

print(f"\nBag of Words Matrix (each number = word count in that document):\n")
print(bow_df.to_string())

# Show the fundamental flaw
print("\n" + "=" * 60)
print("⚠️  THE FLAW: Bag of Words ignores word order!\n")

order_examples = ["dog bites man", "man bites dog"]
order_vec = CountVectorizer()
order_matrix = order_vec.fit_transform(order_examples)

for i, doc in enumerate(order_examples):
    print(f"  \"{doc}\" → {order_matrix.toarray()[i]}")

print(f"\n  Identical vectors! But 'dog bites man' and 'man bites dog'")
print(f"  are VERY different stories.")

Documents:
  Doc 1: "I love this movie it is great"
  Doc 2: "This movie is terrible I hate it"
  Doc 3: "Great movie with great acting"

Bag of Words Matrix (each number = word count in that document):

       acting  great  hate  is  it  love  movie  terrible  this  with
Doc 1       0      1     0   1   1     1      1         0     1     0
Doc 2       0      0     1   1   1     0      1         1     1     0
Doc 3       1      2     0   0   0     0      1         0     0     1

⚠️  THE FLAW: Bag of Words ignores word order!

  "dog bites man" → [1 1 1]
  "man bites dog" → [1 1 1]

  Identical vectors! But 'dog bites man' and 'man bites dog'
  are VERY different stories.


## 3.2 TF-IDF

TF-IDF (Term Frequency–Inverse Document Frequency) improves on raw counts by weighing words based on how **unique** they are across all documents. Common words get lower scores; rare, distinctive words get higher scores.

In [15]:
# Same documents, but with TF-IDF scoring
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray().round(3),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(documents))]
)

print("TF-IDF Matrix (higher score = more important to that document):\n")
print(tfidf_df.to_string())

print("\nNotice:")
print("  - Words like 'movie', 'is', 'this' appear in multiple docs → LOWER scores")
print("  - Words like 'love', 'hate', 'terrible' are unique to one doc → HIGHER scores")
print("  - TF-IDF automatically highlights what makes each document distinctive!")

TF-IDF Matrix (higher score = more important to that document):

       acting  great   hate     is     it   love  movie  terrible   this   with
Doc 1   0.000  0.397  0.000  0.397  0.397  0.523  0.309     0.000  0.397  0.000
Doc 2   0.000  0.000  0.495  0.376  0.376  0.000  0.292     0.495  0.376  0.000
Doc 3   0.463  0.704  0.000  0.000  0.000  0.000  0.274     0.000  0.000  0.463

Notice:
  - Words like 'movie', 'is', 'this' appear in multiple docs → LOWER scores
  - Words like 'love', 'hate', 'terrible' are unique to one doc → HIGHER scores
  - TF-IDF automatically highlights what makes each document distinctive!


## 3.3 N-grams: Capturing Word Pairs

Single words (unigrams) miss important phrases. N-grams capture sequences of words together.

In [16]:
# The problem with single words
review = "The food was not good and the service was not great"

# Unigrams only
unigram_vec = CountVectorizer(ngram_range=(1, 1))
unigram_result = unigram_vec.fit_transform([review])
print("Unigrams (single words):")
print(f"  Features: {unigram_vec.get_feature_names_out().tolist()}")
print(f"  Problem: 'not' and 'good' are separate — we lose 'not good' as a unit\n")

# Bigrams capture word pairs
bigram_vec = CountVectorizer(ngram_range=(1, 2))
bigram_result = bigram_vec.fit_transform([review])
print("Unigrams + Bigrams (single words AND word pairs):")
features = bigram_vec.get_feature_names_out().tolist()
print(f"  Features: {features}")
print(f"\n  Now we capture 'not good' and 'not great' as features!")
print(f"  This helps the model understand negation.")

Unigrams (single words):
  Features: ['and', 'food', 'good', 'great', 'not', 'service', 'the', 'was']
  Problem: 'not' and 'good' are separate — we lose 'not good' as a unit

Unigrams + Bigrams (single words AND word pairs):
  Features: ['and', 'and the', 'food', 'food was', 'good', 'good and', 'great', 'not', 'not good', 'not great', 'service', 'service was', 'the', 'the food', 'the service', 'was', 'was not']

  Now we capture 'not good' and 'not great' as features!
  This helps the model understand negation.


---

# Part 4: Building a Sentiment Classifier

**Scenario:** Your company wants to automatically classify incoming product reviews as positive or negative. Let's build a real classifier and see how it handles easy cases, hard cases, and the tricky examples from Part 1.

## 4.1 Training Data

We need labeled examples for the model to learn from.

In [17]:
# Our labeled training data — positive and negative reviews
reviews = [
    # Positive reviews
    "This product is absolutely fantastic I love it",
    "Best purchase I have ever made highly recommend",
    "Amazing quality and fast shipping very happy",
    "Love this so much it exceeded my expectations",
    "Wonderful product great value for the price",
    "Excellent customer service and beautiful design",
    "Perfect gift my friend loved it thanks",
    "Outstanding quality I will buy again for sure",
    
    # Negative reviews
    "Terrible product broke after one week waste of money",
    "Worst purchase ever do not buy this junk",
    "Awful quality and horrible customer service",
    "Very disappointing does not work as described",
    "Cheap materials and bad design total waste",
    "Hate this product returning it immediately",
    "Poor quality and arrived damaged very upset",
    "Useless product nothing like the advertisement",
]

labels = ["positive"] * 8 + ["negative"] * 8

print(f"Training data: {len(reviews)} reviews ({labels.count('positive')} positive, {labels.count('negative')} negative)")
print(f"\nSample positive: \"{reviews[0]}\"")
print(f"Sample negative: \"{reviews[8]}\"")

Training data: 16 reviews (8 positive, 8 negative)

Sample positive: "This product is absolutely fantastic I love it"
Sample negative: "Terrible product broke after one week waste of money"


## 4.2 Train the Classifier

We'll use a **pipeline** that combines vectorization and classification in one step.

In [22]:
# Build a pipeline: text → Bag of Words → Naive Bayes classifier
sentiment_pipeline = Pipeline([
    ('vectorizer', CountVectorizer()),       # Convert text to numbers
    ('classifier', MultinomialNB()),          # Learn patterns
])

# Train the model
sentiment_pipeline.fit(reviews, labels)

print("Model trained!")
print(f"Vocabulary size: {len(sentiment_pipeline['vectorizer'].get_feature_names_out())} unique words")
print(f"\nThe model has learned associations between words and sentiment.")
print(f"Now let's see how well it actually works...")

Model trained!
Vocabulary size: 81 unique words

The model has learned associations between words and sentiment.
Now let's see how well it actually works...


## 4.3 Test on Easy Cases

First, let's try some clearly positive and negative reviews.

In [23]:
easy_tests = [
    ("I love this product it is amazing",              "positive"),
    ("This is the best thing I have ever bought",      "positive"),
    ("Horrible experience would not recommend",        "negative"),
    ("Terrible quality very disappointed",             "negative"),
]

print("=== Easy Cases ===\n")
for text, expected in easy_tests:
    prediction = sentiment_pipeline.predict([text])[0]
    match = "✅" if prediction == expected else "❌"
    print(f"  {match} \"{text}\"")
    print(f"     Predicted: {prediction} | Expected: {expected}\n")

=== Easy Cases ===

  ✅ "I love this product it is amazing"
     Predicted: positive | Expected: positive

  ✅ "This is the best thing I have ever bought"
     Predicted: positive | Expected: positive

  ✅ "Horrible experience would not recommend"
     Predicted: negative | Expected: negative

  ✅ "Terrible quality very disappointed"
     Predicted: negative | Expected: negative



## 4.4 Test on Hard Cases

Now let's throw the tricky stuff at it — sarcasm, slang, ambiguity, and negation. These are the challenges we explored in Part 1.

In [24]:
hard_tests = [
    # Sarcasm
    ("Oh great another product that breaks immediately",   "negative",  "Sarcasm"),
    ("Wonderful just wonderful my package never arrived",  "negative",  "Sarcasm"),
    ("What a fantastic waste of my hard earned money",     "negative",  "Sarcasm"),
    
    # Negation
    ("This product is not good",                           "negative",  "Negation"),
    ("I would not recommend this to anyone",               "negative",  "Negation"),
    
    # Slang
    ("This product is mid honestly",                       "negative",  "Slang"),
    ("Ngl this thing is fire",                             "positive",  "Slang"),
    
    # Ambiguity
    ("The operation was a success",                        "positive",  "Ambiguity"),
    ("It is what it is",                                   "neutral",   "Ambiguity"),
]

print("=== Hard Cases — Where Simple Models Struggle ===\n")
correct = 0
for text, expected, category in hard_tests:
    prediction = sentiment_pipeline.predict([text])[0]
    match = "✅" if prediction == expected else "❌"
    if prediction == expected:
        correct += 1
    print(f"  {match} [{category}] \"{text}\"")
    print(f"     Predicted: {prediction} | Expected: {expected}")
    if prediction != expected:
        print(f"     → The model got fooled!")
    print()

print(f"Hard case accuracy: {correct}/{len(hard_tests)}")
print(f"\nThis is why NLP is hard — and why we need more sophisticated approaches!")

=== Hard Cases — Where Simple Models Struggle ===

  ✅ [Sarcasm] "Oh great another product that breaks immediately"
     Predicted: negative | Expected: negative

  ❌ [Sarcasm] "Wonderful just wonderful my package never arrived"
     Predicted: positive | Expected: negative
     → The model got fooled!

  ✅ [Sarcasm] "What a fantastic waste of my hard earned money"
     Predicted: negative | Expected: negative

  ✅ [Negation] "This product is not good"
     Predicted: negative | Expected: negative

  ✅ [Negation] "I would not recommend this to anyone"
     Predicted: negative | Expected: negative

  ❌ [Slang] "This product is mid honestly"
     Predicted: positive | Expected: negative
     → The model got fooled!

  ✅ [Slang] "Ngl this thing is fire"
     Predicted: positive | Expected: positive

  ❌ [Ambiguity] "The operation was a success"
     Predicted: negative | Expected: positive
     → The model got fooled!

  ❌ [Ambiguity] "It is what it is"
     Predicted: positive | Expected

## 4.5 The Takeaway

Let's recap what we've learned from building this classifier.

In [25]:
print("=" * 60)
print("WHAT WE LEARNED TODAY")
print("=" * 60)

print("""
✅ WHAT WORKS:
   • Text cleaning pipeline (lowercase, remove noise, tokenize)
   • Bag of Words and TF-IDF for converting text to numbers
   • Simple classifiers work well on straightforward cases
   • N-grams help capture some multi-word patterns

❌ WHAT BREAKS:
   • Sarcasm — positive words with negative intent
   • Slang — words the model has never seen before
   • Negation — "not good" gets split into "not" + "good"
   • Context — same words, different meanings
   • Word order — Bag of Words can't tell the difference

🔮 WHAT'S NEXT:
   • Word embeddings — representing words as vectors that
     capture MEANING, not just counts
   • Text vectorization — "king - man + woman = queen"
   • Building toward TRANSFORMER ARCHITECTURE —
     the technology behind ChatGPT, BERT, and modern AI
""")

print("The simple approaches we learned today are the FOUNDATION.")
print("Every advanced NLP system builds on these exact concepts.")
print("Next class, we level up. 🚀")

WHAT WE LEARNED TODAY

✅ WHAT WORKS:
   • Text cleaning pipeline (lowercase, remove noise, tokenize)
   • Bag of Words and TF-IDF for converting text to numbers
   • Simple classifiers work well on straightforward cases
   • N-grams help capture some multi-word patterns

❌ WHAT BREAKS:
   • Sarcasm — positive words with negative intent
   • Slang — words the model has never seen before
   • Negation — "not good" gets split into "not" + "good"
   • Context — same words, different meanings
   • Word order — Bag of Words can't tell the difference

🔮 WHAT'S NEXT:
   • Word embeddings — representing words as vectors that
     capture MEANING, not just counts
   • Text vectorization — "king - man + woman = queen"
   • Building toward TRANSFORMER ARCHITECTURE —
     the technology behind ChatGPT, BERT, and modern AI

The simple approaches we learned today are the FOUNDATION.
Every advanced NLP system builds on these exact concepts.
Next class, we level up. 🚀
